# 06 Feature engineering

Build one feature row per retained transaction using the [feature audit](../reports/feature_audit.csv), the [training EDA](../reports/eda_findings.md), and the [verified integration](../reports/data_integration.md). The EDA found weak individual relationships with fraud, so these features are candidates for later validation. This notebook does not load labels or fit preprocessing.

The prediction clock is the recorded transaction timestamp. Current-event inputs are eligible when their readings or flags have arrived by scoring. The offline release has no separate arrival clock. Fixed storage specifications were verified for TRACE-X by the integration phase.

## Feature definitions and availability

| Feature | Calculation | Why it might help | Available when | Leakage check |
| --- | --- | --- | --- | --- |
| `event_type` | Copy the current lifecycle step. | Distinguishes shipment, receipt and other stages. | When the current event is recorded. | Use the current step only. Do not infer later steps. |
| `leg` | Copy the current recorded leg. | Gives the position of the current shipment leg. | When the leg is recorded on this event. | Do not use the final number of legs. |
| `humidity` | Copy the current humidity reading. | Gives environmental context. | When the current reading arrives. | Late readings are ineligible at this score. |
| `weight` | Copy the current weight reading. | Gives context for unusual weight readings. | When the current reading arrives. | Preserve valid outliers. Use no later weigh-in. |
| `battery_level` | Copy the current battery reading. | Helps describe sensor condition. | When the current reading arrives. | Use no later reading or end-of-trip status. |
| `rfid_scanned` | Copy the current RFID scan flag. | Describes whether a check was recorded. | When the current scan result arrives. | Do not substitute a later scan outcome. |
| `door_open` | Copy the current door sensor flag. | Describes handling at this event. | When the current flag arrives. | Use no later door activity. |
| `is_cold_chain` | Copy the verified fixed catalogue flag. | Distinguishes products requiring cold handling. | From the fixed catalogue linked to the batch at scoring in TRACE-X. | A live system would need the catalogue version known at scoring. |
| `storage_range_width` | Subtract `storage_min` from `storage_max`. | Describes how narrow the product's permitted storage range is. | When the verified catalogue specifications are known. | Use no later specification. This may overlap with the cold-chain flag. |
| `prior_event_count` | Count events from this batch with times strictly before the score. | Measures how much batch history has accumulated. | At the current event, from prior recorded events. | Never use the final count, current row, or timestamp ties. |
| `seconds_since_prior_event` | Subtract the most recent strictly earlier event time from the scoring time. | Describes an unusually short or long gap. | Once the scoring time and previous event time are known. | Must remain missing if there is no earlier event. The next event is never used. |
| `seconds_since_first_prior_event` | Subtract the earliest strictly earlier event time from the scoring time. | Measures elapsed time since the first observed batch event. | Once the scoring time and earliest earlier event are known. | Use observed history only. Keep it missing on the first timestamp group. |
| `prior_mean_humidity` | Average humidity over strictly earlier events of this batch. | Establishes context for the current reading. | From earlier completed records available at scoring. | No current reading or future batch average. |
| `prior_mean_weight` | Average weight over strictly earlier events of this batch. | Gives a past reference for the current weight. | From earlier completed records available at scoring. | No current weight or full-batch average. |
| `prior_rfid_scan_rate` | Divide earlier true RFID flags by the earlier event count. | Describes past scan coverage for this batch. | From earlier completed records available at scoring. | No current or later flags and no fraud outcomes. |
| `prior_door_open_rate` | Divide earlier true door flags by the earlier event count. | Describes past handling for this batch. | From earlier completed records available at scoring. | No current or later flags and no fraud outcomes. |

The raw clock and batch ID are metadata. Raw calendar features can reflect simulator campaigns. Quantity, unverified catalogue attributes, changing entity data, simulator shortcuts and fraud outcomes remain outside the feature list. Product history across batches would require a verified product identity and data available at the event time. This phase uses fixed product specifications and within-batch history.

In [ ]:
from hashlib import file_digest
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import yaml
from IPython.display import display

current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True
from src.features import FEATURE_COLUMNS, HISTORY_FEATURES, OUTPUT_COLUMNS, build_features
from src.integration import (
    BATCH_READ_COLUMNS, ENTITY_READ_COLUMNS, MEDICINE_READ_COLUMNS, PREDICTION_TIMESTAMP,
    TRANSACTION_KEY, TRANSACTION_READ_COLUMNS, EVENT_FEATURES, MEDICINE_FEATURES,
    integrate_transactions, require_unique_key,
)

processed = root / "processed"
cleaned = processed / "cleaned"
boundary = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundary["assignment_file"]
with manifest_path.open("rb") as source:
    assert file_digest(source, "sha256").hexdigest() == boundary["assignment_sha256"]
assert list(TRANSACTION_KEY) == boundary["transaction_key"]
assert PREDICTION_TIMESTAMP == boundary["prediction_timestamp"]

audit = pd.read_csv(root / "reports" / "feature_audit.csv")
approved = audit.loc[audit["category"].eq("Safe input features")]
assert set(EVENT_FEATURES) == set(approved.loc[
    approved["table"].eq("transactions") & approved["planned_use"].isin(["current_event_input", "current_event_input_and_key"]),
    "column",
])
assert set(MEDICINE_FEATURES) <= set(approved.loc[
    approved["table"].eq("medicines") & approved["planned_use"].eq("conditional_lookup"), "column"
])
config = yaml.safe_load((root / "Phase1_Dataset/code/config.yaml").read_text(encoding="utf-8"))
print("Checked the approved sources and the frozen split manifest.")

## Run leakage tests

The tests compare earlier output before and after modifying or adding later events. They also change readings on simultaneous rows and check that none of those readings enters the history of another row at that timestamp. A small hand-calculated example checks elapsed times, averages, rates and isolation between batches.

In [ ]:
completed = subprocess.run(
    [sys.executable, "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_features.py"],
    cwd=root, capture_output=True, text=True,
)
print(completed.stdout, end="")
assert completed.returncode == 0, completed.stderr or completed.stdout

## Process the observed event stream in time order

Rebuild the verified integration in memory. Match the frozen manifest on the complete transaction key and timestamp. Use the eligible training, validation and test events in one chronological pass. Boundary-spanning batches stay excluded as recorded by the manifest. Since the frozen split assigns each batch to one set, each retained batch history stays within that same set.

The history function groups events by batch and timestamp. It records the previous cumulative sums before exposing any events at the current time to later scores. All tied rows receive the same history. Predictions at the first timestamp of each batch get a count of zero and missing values for history that does not yet exist.

In [ ]:
def read_cleaned(name, columns):
    return pd.read_parquet(cleaned / f"{name}.parquet", columns=list(columns))


# The integration checks catalogue provenance using audit-only fields. They never enter its output.
integrated, joins, timing = integrate_transactions(
    read_cleaned("transactions", TRANSACTION_READ_COLUMNS),
    read_cleaned("batches", BATCH_READ_COLUMNS),
    read_cleaned("medicines", MEDICINE_READ_COLUMNS),
    read_cleaned("entities", ENTITY_READ_COLUMNS),
    config["storage_ranges"],
)
assert timing["storage_rate_matches"] == len(integrated)
clock = PREDICTION_TIMESTAMP
key = list(TRANSACTION_KEY)
identity = [*key, clock]
assignments = pd.read_parquet(manifest_path, columns=[*identity, "split"])
require_unique_key(assignments, key, "fixed assignments")
assigned = integrated.merge(assignments, on=identity, how="inner", validate="one_to_one")
assert len(assigned) == len(assignments) == len(integrated) == boundary["source_rows"]
del assignments, integrated

split_names = ("train", "validation", "test")
assert set(assigned["split"]) == {*split_names, boundary["excluded_assignment"]}
assert assigned.groupby("batch_id")["split"].nunique().eq(1).all()
eligible = assigned.loc[assigned["split"].isin(split_names)]
excluded_rows = len(assigned) - len(eligible)
assert excluded_rows == boundary["split_summary"][boundary["excluded_assignment"]]["rows"]
del assigned

engineered = build_features(eligible)
# Recover assignment metadata in the exact chronological order returned by the feature function.
chronological = eligible[[*identity, "split"]].sort_values([clock, *key], ignore_index=True)
pd.testing.assert_frame_equal(engineered[identity], chronological[identity])
engineered["split"] = chronological["split"].to_numpy()
del chronological, eligible
print(f"Built {len(engineered):,} feature rows. The frozen manifest excludes {excluded_rows:,} boundary rows.")

## Validate the full result

Independently count prior rows per batch by subtracting the row's position within its timestamp tie group from its position within the batch. Compare that count to the engineered value on every row. Also verify chronological split order, positive gaps, valid flag rates, and missing history only when there are no prior events. The display reports structural counts only.

In [ ]:
assert engineered.columns.tolist() == [*OUTPUT_COLUMNS, "split"]
assert len(engineered) == boundary["retained_rows"]
assert engineered[clock].is_monotonic_increasing
require_unique_key(engineered, key, "engineered transactions")
assert boundary["target_column"] not in engineered
assert set(FEATURE_COLUMNS).isdisjoint({"batch_id", clock, "split"})

earlier = engineered.groupby("batch_id", sort=False).cumcount()
earlier -= engineered.groupby(["batch_id", clock], sort=False).cumcount()
assert engineered["prior_event_count"].eq(earlier).all()
no_history = earlier.eq(0)
history_values = list(HISTORY_FEATURES[1:])
assert engineered.loc[no_history, history_values].isna().all().all()
assert engineered.loc[~no_history, history_values].notna().all().all()
assert engineered.loc[~no_history, ["seconds_since_prior_event", "seconds_since_first_prior_event"]].gt(0).all().all()
assert engineered.loc[~no_history, "seconds_since_first_prior_event"].ge(
    engineered.loc[~no_history, "seconds_since_prior_event"]
).all()
for field in ("prior_rfid_scan_rate", "prior_door_open_rate"):
    assert engineered.loc[~no_history, field].between(0, 1).all()

summary = engineered.assign(no_history=no_history).groupby("split").agg(
    transactions=(clock, "size"),
    first_time=(clock, "min"),
    last_time=(clock, "max"),
    rows_without_prior_events=("no_history", "sum"),
    max_prior_events=("prior_event_count", "max"),
).loc[list(split_names)]
assert summary["transactions"].to_dict() == {
    name: boundary["split_summary"][name]["rows"] for name in split_names
}
assert summary.loc["train", "last_time"] < summary.loc["validation", "first_time"]
assert summary.loc["validation", "last_time"] < summary.loc["test", "first_time"]
display(summary)
print(f"Verified {len(FEATURE_COLUMNS)} candidate inputs and all full-release feature assertions.")

The `engineered` DataFrame contains the verified feature rows and the frozen split marker in memory. Reuse the feature list in `src/features.py` so that the marker, identifiers and clock stay outside later model inputs. The target remains in its separate artifact.